In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

sys.path.append("../..")
from src.data.dataset_utils import exclude_boundary_nodes
from src.utils.plot_functions import plot_overall_scores, plot_binned_scores

VARIABLE = "sigWaveHeight"  # significant wave height

PREDICT_FINAL_DIR = Path("/work/ka1176/caroline/gitlab/wave-wind-model-emulator/logs/predict_final")
GROUND_TRUTH_ZARR = "/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_RUN04b_JUWELS_wpile.zarr"
BOUNDARY_NODES_CSV = "../../src/data/node_selection/boundary_nodes.csv"

# produced by notebooks/paper-plots/compute_experiment_metrics.py
METRICS_DIR = Path("metrics")

CHUNKS = "auto"  # keep everything dask-backed/lazy; set to None to load eagerly

In [ ]:
PAPER_WIDE = (7.1, 3.5)

In [ ]:
def load_experiment_predictions(exp_dir: Path, variable: str = VARIABLE, chunks=CHUNKS) -> xr.Dataset | None:
    """Load and concatenate the summer/winter prediction files of one experiment folder.

    Returns None if no `predictions_*.nc` files are found yet in `exp_dir`.
    """
    nc_files = sorted(exp_dir.glob("predictions_*.nc"))
    if not nc_files:
        return None
    parts = [xr.open_dataset(f, engine="netcdf4", chunks=chunks)[[variable]] for f in nc_files]
    return xr.concat(parts, dim="time").sortby("time")


def open_predict_final_experiments(
    predict_final_dir: Path = PREDICT_FINAL_DIR,
    ground_truth: str = GROUND_TRUTH_ZARR,
    variable: str = VARIABLE,
    chunks=CHUNKS,
) -> xr.Dataset:
    """Merge predictions from all experiment subfolders with ground truth ("true") along an
    "experiment" dimension. Experiment folders without any prediction file yet are skipped.
    """
    exp_datasets = {}
    for exp_dir in sorted(p for p in predict_final_dir.iterdir() if p.is_dir()):
        ds_exp = load_experiment_predictions(exp_dir, variable=variable, chunks=chunks)
        if ds_exp is None:
            print(f"Skipping '{exp_dir.name}': no prediction files found yet")
            continue
        print(f"Loaded '{exp_dir.name}' ({ds_exp.time.values[0]} .. {ds_exp.time.values[-1]}, {ds_exp.sizes['time']} steps)")
        exp_datasets[exp_dir.name] = ds_exp

    if not exp_datasets:
        raise RuntimeError(f"No prediction files found under {predict_final_dir}")

    exp_names = list(exp_datasets.keys())
    ds_pred = xr.concat(exp_datasets.values(), dim=pd.Index(exp_names, name="experiment"))

    # ground truth, restricted to the time steps covered by the predictions
    ds_true = xr.open_zarr(ground_truth, chunks=chunks)[[variable]]
    ds_true = ds_true.sel(time=ds_pred.time.values).expand_dims(experiment=["true"])

    return xr.concat([ds_pred, ds_true], dim="experiment")


def parse_bin_column(bin_series: pd.Series) -> pd.Series:
    """Turn the 'bin' column of a binned-metrics CSV back into pd.Interval objects.

    compute_binned_scores/compute_binned_change_scores return 'bin' as a pd.Interval
    (or the string 'overall'), which round-trips through CSV as a plain string like
    '(0.0, 0.5]'. plot_binned_scores expects real pd.Interval objects (for .mid), so
    we parse them back here after loading the CSV.
    """

    def _parse(value):
        if value == "overall":
            return value
        left, right = value.strip("()[]").split(", ")
        return pd.Interval(float(left), float(right), closed="right")

    return bin_series.apply(_parse)

In [ ]:
ds = open_predict_final_experiments()

In [ ]:
ds = exclude_boundary_nodes(ds, BOUNDARY_NODES_CSV)

In [ ]:
# account for time lag in LSTM models
# Find valid indices along 'time' where space == 's1' is not NaN
valid_times = ds.sel(nSCHISM_hgrid_node=0).notnull()
print(valid_times)

# Apply the boolean mask along the 'time' dimension
result = ds.where(valid_times, drop=True)


In [ ]:
print(ds.experiment.values)
print(ds.sizes)

print(ds[VARIABLE].mean(dim=("time", "nSCHISM_hgrid_node")).compute())

## RMSE

### Global

In [ ]:
summary_df = pd.read_csv(METRICS_DIR / "summary_metrics.csv")
summary_df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)
sns.barplot(data=summary_df, x="experiment", y="total_rmse", ax=ax)
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

### Summer

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)
sns.barplot(data=summary_df, x="experiment", y="summer_rmse", ax=ax)
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

### Winter

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)
sns.barplot(data=summary_df, x="experiment", y="winter_rmse", ax=ax)
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

### Bias

In [ ]:
bias_df = summary_df.melt(
    id_vars="experiment",
    value_vars=["total_bias", "summer_bias", "winter_bias"],
    var_name="period",
    value_name="bias",
)
bias_df["period"] = bias_df["period"].str.replace("_bias", "")

fig, ax = plt.subplots(1, 1, figsize=PAPER_WIDE)
sns.barplot(data=bias_df, x="experiment", y="bias", hue="period", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

### Binned by value

In [ ]:
binned_value_df = pd.read_csv(METRICS_DIR / "binned_value_metrics.csv")
binned_value_df["bin"] = parse_bin_column(binned_value_df["bin"])

plot_overall_scores(binned_value_df, metric="rmse")
plot_binned_scores(binned_value_df, metric="rmse")

### Binned by change

In [ ]:
binned_change_df = pd.read_csv(METRICS_DIR / "binned_change_metrics.csv")
binned_change_df["bin"] = parse_bin_column(binned_change_df["bin"])

plot_overall_scores(binned_change_df, metric="rmse")
plot_binned_scores(binned_change_df, metric="rmse")

## Subgraph effects